<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/Baseline-v7.5.3---Two-Prompt-Iteration/mnps_new_baseline%20v8.0.0TP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MNPS Job Equity New Baseline 8.0.0TP**
> A notebook to help you get started  
> DSI DSSG + MNPS   

> # **Version 8.0.0TP Changes**
> - **Two-Part Dialogue Structure**
> - Part 1: Forces analysis conversation before classification
> - Part 2: Structured output with enhanced decision frameworks


In [ ]:
# ==== 1) Imports, paths, inputs from v7.5.3 artifacts ====
import os, json, shutil, datetime as dt, zipfile
from pathlib import Path
import pandas as pd
import numpy as np
import re
import time
import random
from google.colab import drive
from openai import OpenAI

# Mount Google Drive
drive.mount('/content/drive')

# Create unique run folder
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder = f"RUN_{timestamp}"
base_path = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
run_path = base_path / run_folder
run_path.mkdir(parents=True, exist_ok=True)

# Create outputs subfolder
OUTPUTS_DIR = run_path / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Run folder: {run_path}")
print(f"📁 Outputs dir: {OUTPUTS_DIR}")

# Load all required files - Updated for Colab root path
RUN_ROOT = Path('/content')

# Unzip MNPS Prompt Resources if needed
ZIP_FILE = RUN_ROOT / "MNPS Prompt Resources.zip"
if ZIP_FILE.exists():
    print(f"📦 Found {ZIP_FILE}, extracting...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_ref.extractall(RUN_ROOT)
    print("✅ Extracted MNPS Prompt Resources")
else:
    print("⚠️  MNPS Prompt Resources.zip not found - make sure to upload it")

# Core data files
BATCH_INPUT_CSV = RUN_ROOT / "Sample JDs.csv"
GT_MASTERFILE_CSV = RUN_ROOT / "Ground Truth Masterfile.csv"

# MNPS Prompt Resources (from extracted zip)
MNPS_ROLES_CSV = RUN_ROOT / "MNPS Roles.csv"
MNPS_KSACS_CSV = RUN_ROOT / "MNPS KSACs.csv"
COMPETENCY_EXTENDED_CSV = RUN_ROOT / "Competency Extended Descriptions.csv"
KORN_FERRY_CSV = RUN_ROOT / "Korn_Ferry Lominger 38 Competencies.csv"

print(f"📄 Batch input: {BATCH_INPUT_CSV}")
print(f"📄 Ground truth: {GT_MASTERFILE_CSV}")
print(f"📄 MNPS roles: {MNPS_ROLES_CSV}")
print(f"📄 MNPS KSACs: {MNPS_KSACS_CSV}")
print(f"📄 Competency Extended: {COMPETENCY_EXTENDED_CSV}")
print(f"📄 Korn Ferry: {KORN_FERRY_CSV}")

# Load data
df = pd.read_csv(BATCH_INPUT_CSV, encoding='latin1')
gt_df = pd.read_csv(GT_MASTERFILE_CSV, encoding='latin1')
roles_df = pd.read_csv(MNPS_ROLES_CSV, encoding='latin1')
ksacs_df = pd.read_csv(MNPS_KSACS_CSV, encoding='latin1')
competency_df = pd.read_csv(COMPETENCY_EXTENDED_CSV, encoding='latin1')
korn_ferry_df = pd.read_csv(KORN_FERRY_CSV, encoding='latin1')

print(f"✅ Loaded {len(df)} job descriptions")
print(f"✅ Loaded {len(gt_df)} ground truth records")
print(f"✅ Loaded {len(roles_df)} MNPS roles")
print(f"✅ Loaded {len(ksacs_df)} MNPS KSACs")
print(f"✅ Loaded {len(competency_df)} competency descriptions")
print(f"✅ Loaded {len(korn_ferry_df)} Korn Ferry competencies")


In [ ]:
# ==== 2) Load data and build attribute-only view (ignore title) ====

# Load prediction data (if available from previous runs)
# For now, we'll work with the raw job descriptions
preds = df.copy()

# Build attribute-only text (ignore job titles)
ATTR_COLS = [
    'Position Summary', 'Essential Functions', 'Work Experience', 'Education',
    'Licenses and Certifications', 'Knowledge, Skills and Abilities'
]

# Combine all attribute text
attrs = df[ATTR_COLS].fillna('')
text = attrs['Position Summary'] + ' ' + attrs['Essential Functions'] + ' ' + \
       attrs['Work Experience'] + ' ' + attrs['Education'] + ' ' + \
       attrs['Licenses and Certifications'] + ' ' + attrs['Knowledge, Skills and Abilities']

print(f"✅ Built attribute-only view for {len(text)} job descriptions")
print(f"✅ Ignoring job titles - focusing on job attributes only")


In [ ]:
# ==== 3) Enhanced closed sets and normalization helpers with Problem Role Cheat Sheet logic ====

# Get MNPS roles from the loaded data
# Handle different possible column names
role_columns = [col for col in roles_df.columns if 'role' in col.lower()]
if role_columns:
    VALID_ROLES = roles_df[role_columns[0]].dropna().tolist()
else:
    # Fallback to first column
    VALID_ROLES = roles_df.iloc[:, 0].dropna().tolist()

print(f"✅ Found {len(VALID_ROLES)} MNPS roles")

# Enhanced closed sets for major and minor role groups
MAJOR_ALLOWED = [
    'Technician', 'Specialist', 'Analyst', 'Manager', 'Coordinator', 'Director', 'Other',
    'Teacher', 'Coach', 'Counselor', 'Clerical Support', 'Instructor', 'Driver',
    'Supervisor', 'Accountant', 'Architect (Facility-Focused)', 'Architect (Technology-Focused)',
    'Principal', 'Librarian', 'Social Worker', 'Therapist', 'Translator', 'Skilled Laborer',
    'Administrative Assistant'
]

MINOR_ALLOWED = ['I', 'II', 'III', 'Lead']

# Normalization mapping for minor roles
CANON_MINOR_MAP = {
    'i': 'I', '1': 'I', 'one': 'I', 'entry': 'I',
    'ii': 'II', '2': 'II', 'two': 'II',
    'iii': 'III', '3': 'III', 'three': 'III',
    'lead': 'Lead', 'iv': 'III', '4': 'III'
}

# Enhanced specialist fallback patterns with Problem Role Cheat Sheet logic
SPECIALIST_FALLBACKS = [
    # Technician patterns - hands-on technical work, equipment, maintenance
    ('Technician', 'technical|repair|maintenance|install|troubleshoot|equipment|hands-on|tools|machinery|systems'),

    # Analyst patterns - data analysis, research, evaluation
    ('Analyst', 'analyze|data analysis|research|evaluate|assess|statistical|quantitative|qualitative|metrics|reports'),

    # Teacher patterns - classroom instruction, curriculum, students
    ('Teacher', 'classroom|lesson|instruction|teacher|students|curriculum|teaching|educational|academic'),

    # Coach patterns - mentoring, professional development, instructional support
    ('Coach', 'coach|instructional coach|plc|model lessons|co-teach|mentor|professional development|instructional support'),

    # Clerical Support patterns - administrative, office work, records
    ('Clerical Support', 'clerk|clerical|records|data entry|office support|administrative|filing|correspondence'),

    # Counselor patterns - guidance, therapy, mental health
    ('Counselor', 'counsel|social-emotional|guidance|therapy|mental health|behavioral|psychological'),

    # Manager patterns - management, supervision, strategic planning
    ('Manager', 'manage|supervise|budget|oversight|lead team|program manager|direct|strategic|planning|policy'),

    # Accountant patterns - financial, accounting, bookkeeping
    ('Accountant', 'accounting|financial|bookkeeping|audit|budget|finance|accounts payable|accounts receivable|fiscal'),

    # Coordinator patterns - coordination, organization, facilitation
    ('Coordinator', 'coordinate|organize|facilitate|liaison|program coordination|project coordination|event coordination'),

    # Architect patterns - building/construction vs technology
    ('Architect (Facility-Focused)', 'building|construction|facility|architectural|design|space planning|renovation|infrastructure'),
    ('Architect (Technology-Focused)', 'system|software|technology|IT|database|network|programming|technical architecture')
]

# Executive roles that rarely have "Lead" minor sub-grouping
EXECUTIVE_ROLES = ['Coordinator', 'Principal', 'Director', 'Manager']

def normalize_minor(x: str) -> str:
    """Normalize minor role to approved values."""
    if pd.isna(x):
        return 'I'
    s = str(x).strip()
    if s in MINOR_ALLOWED:
        return s
    s_low = s.lower()
    return CANON_MINOR_MAP.get(s_low, 'I')

def discourage_specialist(text: str, proposed_major: str) -> str:
    """Enhanced logic to discourage overuse of 'Specialist' based on Problem Role Cheat Sheet."""
    if proposed_major != 'Specialist':
        return proposed_major

    t = (text or '').lower()

    # Check for more specific role matches first
    for major, pattern in SPECIALIST_FALLBACKS:
        if re.search(pattern, t):
            return major

    # If no specific match, return Specialist
    return proposed_major

def distinguish_supervisor_manager(text: str, proposed_major: str) -> str:
    """Distinguish between Supervisor and Manager based on education requirements.

    Supervisor: Primarily manages people, no post-high school education required
    Manager: Does more than manage people, requires minimum associates degree
    """
    if proposed_major not in ['Supervisor', 'Manager']:
        return proposed_major

    t = (text or '').lower()

    # Check for education requirements
    has_degree_requirement = re.search(r'(associate|bachelor|master|degree|college)', t)

    # Check for broader responsibilities beyond people management
    has_broader_responsibilities = re.search(r'(budget|strategic|policy|program|project|planning|analysis)', t)

    # If has degree requirement or broader responsibilities, likely Manager
    if has_degree_requirement or has_broader_responsibilities:
        return 'Manager'

    # If primarily people management without degree requirements, likely Supervisor
    if re.search(r'(supervise|oversee|direct|lead team|staff management)', t):
        return 'Supervisor'

    return proposed_major

def refine_coordinator_coach_manager(text: str, proposed_major: str) -> str:
    """Refine distinctions between Coordinator, Coach, and Manager based on Problem Role Cheat Sheet."""
    if proposed_major not in ['Coordinator', 'Coach', 'Manager']:
        return proposed_major

    t = (text or '').lower()

    # Coach patterns - instructional support, mentoring, professional development
    if re.search(r'(instructional|mentor|professional development|co-teach|model lessons|plc)', t):
        return 'Coach'

    # Manager patterns - strategic planning, policy, budget, supervision
    if re.search(r'(strategic|policy|budget|supervise|manage|oversight|planning)', t):
        return 'Manager'

    # Coordinator patterns - coordination, organization, facilitation
    if re.search(r'(coordinate|organize|facilitate|liaison|program|project)', t):
        return 'Coordinator'

    return proposed_major

def fix_executive_minor_sub_grouping(major_role: str, minor_role: str) -> str:
    """Fix minor sub-grouping for executive roles - rarely "Lead", usually "I", "II", or "III"."""
    if major_role not in EXECUTIVE_ROLES:
        return minor_role

    # If it's an executive role and currently "Lead", downgrade to "III" or "II"
    if minor_role == 'Lead':
        # Check if it's a very senior executive role that might warrant "III"
        if major_role in ['Director', 'Principal']:
            return 'III'
        else:
            return 'II'

    return minor_role

print("✅ Enhanced closed sets and normalization helpers with Problem Role Cheat Sheet logic defined")


In [ ]:
# ==== 4) Build comprehensive KSACs text from all MNPS resources ====

def build_ksacs_text():
    """Build comprehensive KSACs text from all MNPS resources."""
    ksacs_text = "MNPS Knowledge, Skills, Abilities, and Competencies (KSACs):\n\n"

    # Clean up column names to handle potential whitespace or case issues
    ksacs_df.columns = ksacs_df.columns.str.strip()
    competency_df.columns = competency_df.columns.str.strip()
    korn_ferry_df.columns = korn_ferry_df.columns.str.strip()

    # Add role-specific KSACs
    # Find columns that contain 'Role' and 'KSACs' (case-insensitive and partial match)
    role_col_ksacs = next((col for col in ksacs_df.columns if 'role' in col.lower()), None)
    ksacs_col_ksacs = next((col for col in ksacs_df.columns if 'ksacs' in col.lower()), None)

    if role_col_ksacs and ksacs_col_ksacs:
        for _, row in ksacs_df.iterrows():
            role = row.get(role_col_ksacs, '')
            ksacs = row.get(ksacs_col_ksacs, '')
            if role and ksacs:
                ksacs_text += f"**{role}**:\n{ksacs}\n\n"
    else:
        print("Warning: Could not find 'Role' or 'KSACs' columns in ksacs_df.")

    # Add competency extended descriptions
    # Find columns that contain 'Competency' and 'Description' (case-insensitive and partial match)
    comp_col_comp = next((col for col in competency_df.columns if 'competency' in col.lower()), None)
    desc_col_comp = next((col for col in competency_df.columns if 'description' in col.lower()), None)

    if comp_col_comp and desc_col_comp:
        ksacs_text += "\n**Competency Extended Descriptions**:\n"
        for _, row in competency_df.iterrows():
            competency = row.get(comp_col_comp, '')
            description = row.get(desc_col_comp, '')
            if competency and description:
                ksacs_text += f"- {competency}: {description}\n"
    else:
        print("Warning: Could not find 'Competency' or 'Description' columns in competency_df.")

    # Add Korn Ferry competencies
    # Find columns that contain 'Competency' and 'Definition' (case-insensitive and partial match)
    comp_col_kf = next((col for col in korn_ferry_df.columns if 'competency' in col.lower()), None)
    def_col_kf = next((col for col in korn_ferry_df.columns if 'description' in col.lower() or 'definition' in col.lower()), None)

    if comp_col_kf and def_col_kf:
        ksacs_text += "\n**Korn Ferry Lominger 38 Competencies**:\n"
        for _, row in korn_ferry_df.iterrows():
            competency = row.get(comp_col_kf, '')
            definition = row.get(def_col_kf, '')
            if competency and definition:
                ksacs_text += f"- {competency}: {definition}\n"
    else:
        print("Warning: Could not find 'Competency' or 'Description'/'Definition' columns in korn_ferry_df.")

    return ksacs_text

KSACS_TEXT = build_ksacs_text()

print(f"✅ Built comprehensive KSACs text ({len(KSACS_TEXT)} characters)")
print("✅ Includes all 4 critical MNPS resource documents")


In [ ]:
# ==== 5) Two-Part Dialogue Prompt v8.0 ====
two_part_dialogue_prompt = """
You are a job classification assistant. This is a two-part dialogue system that helps you classify jobs accurately.

# PART 1: ANALYSIS CONVERSATION
Start with a short two-part dialogue analyzing the job attributes.
- A Conversation Starter provides an ANALYSIS of the job attributes
- A Brief Second Exchange shows pattern recognition

# Format Example:
**Examiner (You):** "Let's analyze this position. The essential functions describe [key work type]. The education requirement is [level], and experience needed is [amount]. The licenses/certifications show [specific credentials or none]. What patterns do we see?"

**Assistant (You continue):** "Based on my analysis:
- Work type: [hands-on technical / analytical / coordination / supervisory / specialized knowledge / instructional]
- Complexity indicators: [education level, years experience, scope of impact]
- Specialized credentials: [yes - specific license/certification name / no specialized credentials]
- Supervision level: [supervises others / no supervision / leads without formal authority]

This aligns with [ROLE] at the [I/II/III/blank] level because..."

---

# PART 2: CLASSIFICATION OUTPUT

After your dialogue analysis, provide this structured output:

## ROLE SELECTION FRAMEWORK

**Priority 1 - Identify Specialized Licensed Roles First:**
- **Therapist**: Physical/Occupational/Speech therapy (requires therapy license) - INCLUDES "Asst Therapy" titles
- **Pathologist (Speech-Language Focused)**: Speech-language pathology
- **Social Worker**: Requires social work license/certification
- **Psychologist**: School psychology license
- **Teacher**: K-12 instruction, requires teaching license, teacher of record (grades/credit, IEP/504)
- **Counselor**: School counseling license
- **Librarian**: Library media specialist certification
- **Principal**: Building leader, requires principal license, supervises ALL staff
- **Assistant Principal**: Assists principal, requires assistant principal license, LIMITED supervision
- **Nurse**: Nursing license

**Priority 2 - Work Type Identification:**

*If hands-on technical/equipment work:*
- **Technician**: Equipment maintenance, troubleshooting, installation, setup, technical support
- **Skilled Laborer**: Traditional trades (plumbing, electrical, carpentry, HVAC, welding, masonry, lawncare)

*If analytical/research work:*
- **Analyst**: Data analysis, research, evaluation, statistical work, reporting, metrics
- **Accountant**: Financial accounting, auditing, general ledger (requires accounting degree + 3+ years)

*If instructional/developmental work (non-licensed):*
- **Instructor**: Adult learning/PD, enrichment clubs, CTE skills labs, JROTC (NO teaching license)
- **Coach**: Teacher support, instructional coaching, professional development cycles, co-teaching
- **Trainer**: Staff training, skill development programs

*If coordination/organizational work:*
- **Coordinator**: Program coordination, logistics, organizing, facilitation, project management (NOT supervision)

*If specialized knowledge (without specialized license):*
- **Specialist**: Deep expertise in specific domain with specialized training/certification (like "Certified Orientation and Mobility Specialist")
- Use ONLY when specialized certifications are present AND no other role fits

*If strategic/supervisory management:*
- **Manager**: Strategic planning, budget oversight, policy development, supervises staff (Bachelor's + 5+ years supervisory)
- **Supervisor**: Direct oversight of day-to-day operations and staff
- **Director**: Department-level leadership, strategic planning, policy setting (usually NO minor sub-group)

*If advisory/consultative work:*
- **Advisor**: Consultative leadership, district-wide strategic guidance, stakeholder engagement

*If administrative/clerical work:*
- **Secretary**: Front-office hub for school/department (phones, visitors, daily operations, scheduling)
- **Administrative Assistant**: Leader/executive enablement (complex calendar, correspondence, briefings)
- **Clerk**: High-accuracy transaction/records processing

*If liaison/representative work:*
- **Liaison**: Connection between groups, relationship management
- **Representative**: Basic liaison duties, facility coordination (high school education level)

**Priority 3 - Avoid These Common Misclassifications:**

❌ **DO NOT classify as Manager when:**
- Role coordinates programs without supervising staff → use **Coordinator**
- Role requires specialized domain knowledge without supervision → use **Specialist**
- Role provides advisory support district-wide → use **Advisor**
- Lacks Bachelor's degree + 5+ years supervisory experience → use **Coordinator** or **Specialist**

❌ **DO NOT classify as Coordinator when:**
- Role provides instructional support to teachers → use **Coach**
- Role has strategic/supervisory leadership → use **Director** or **Manager**

❌ **DO NOT classify as Coach when:**
- Role coordinates programs/logistics (not instructional) → use **Coordinator**

❌ **DO NOT classify as Instructor when:**
- Role is teacher of record with grades/standards/IEP → use **Teacher**

❌ **DO NOT classify as Assistant when:**
- Job title contains "Asst Therapy", "Asst Speech" → use professional role (**Therapist**, **Pathologist**)

❌ **DO NOT classify as Technician when:**
- Role is traditional trade work (plumbing, carpentry) → use **Skilled Laborer**

❌ **DO NOT classify as Accountant when:**
- Only Associates degree or 1-3 years experience → use **Technician**
- True Accountant needs Bachelor's in Accounting + 3+ years

❌ **DO NOT classify as Social Worker/Psychologist when:**
- No specialized license mentioned → use **Liaison** or **Specialist**

## MINOR SUB-GROUP RULES

**MUST be BLANK (no sub-group) for:**
- Teacher
- Librarian
- Counselor
- Principal
- Assistant Principal
- Therapist
- Pathologist (Speech-Language Focused)
- Psychologist
- Social Worker
- Director (usually blank, rarely has sub-group)
- Coach (unless explicitly "Lead Coach")

**Sub-group Definitions:**
- **I**: Entry-level complexity, 1-3 years experience, foundational role
- **II**: Intermediate complexity, 3-5 years experience, broader scope
- **III**: Advanced/senior complexity, 5+ years experience, highest expertise, district-wide impact
- **Lead**: Non-executive team leadership (NOT for principals/directors/executives)

**Sub-group Decision Tree:**
1. Is this Teacher/Librarian/Counselor/Principal/Assistant Principal/Therapist/Coach/Director? → **Blank**
2. Is this entry-level with 1-3 years experience? → **I**
3. Is this intermediate with 3-5 years and expanded scope? → **II**
4. Is this senior-level with 5+ years and district-wide impact? → **III**
5. Does this lead a team without executive authority? → **Lead**

## CRITICAL REMINDERS
- "Asst Therapy Physical" = **Therapist** (not Assistant)
- "Asst Speech" = **Pathologist** (not Assistant)
- "Principal Asst MS" = **Assistant Principal** with NO minor sub-group
- Trade work (plumbing, electrical, carpentry) = **Skilled Laborer**
- Program coordination = **Coordinator** (not Coach or Manager)
- Instructional teacher support = **Coach** (not Coordinator)
- Teacher = teacher of record; Instructor = no teaching license
- Secretary ≠ Administrative Assistant ≠ Clerk (different scopes)
- Manager requires: supervision + strategic planning + Bachelor's + 5+ years supervisory
- Specialist = specialized certifications/training, use sparingly
- Most roles do NOT need minor sub-groups - use blank liberally

"""

print("✅ Two-part dialogue prompt v8.0 defined")

In [ ]:
# ==== 6) OpenAI API Setup with Rate Limiting Protection ====
import os
from google.colab import userdata

# Get API key from Colab's 🔑 panel
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Initialize OpenAI client
client = OpenAI()

# Use GPT-4o-2024-11-20 for stable performance
MODEL_ID = "gpt-4o-2024-11-20"

print(f"✅ OpenAI client initialized")
print(f"✅ Using model: {MODEL_ID}")

def call_llm_json_with_retry(prompt: str, model: str = None, max_retries: int = 3) -> dict:
    """Call OpenAI API with JSON response and exponential backoff for rate limiting."""
    if model is None:
        model = MODEL_ID

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0.2
            )
            return json.loads(response.choices[0].message.content)

        except Exception as e:
            error_str = str(e).lower()

            # Check for rate limiting errors
            if "429" in error_str or "rate limit" in error_str or "quota" in error_str:
                if attempt < max_retries - 1:
                    # Exponential backoff with jitter
                    wait_time = (2 ** attempt) + random.uniform(0, 1)
                    print(f"⚠️  Rate limit hit, waiting {wait_time:.1f} seconds before retry {attempt + 1}/{max_retries}")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"❌ Max retries reached for rate limiting. Error: {e}")
                    raise e
            else:
                # Non-rate limiting error, raise immediately
                print(f"❌ Non-rate limiting error: {e}")
                raise e

    # This should never be reached, but just in case
    raise Exception("Unexpected error in retry logic")

print("✅ call_llm_json_with_retry function defined with rate limiting protection")


In [ ]:
# ==== 7) Modified Batch Processing with Two-Part Dialogue ====
from tqdm import tqdm

def process_job_description_dialogue(row_idx: int, row: pd.Series) -> dict:
    """Process a single job description using two-part dialogue approach."""
    # Build job description text (ignore job title)
    job_text = f"""Position Summary: {row.get('Position Summary', '')}
Essential Functions: {row.get('Essential Functions', '')}
Work Experience: {row.get('Work Experience', '')}
Education: {row.get('Education', '')}
Licenses and Certifications: {row.get('Licenses and Certifications', '')}
Knowledge, Skills and Abilities: {row.get('Knowledge, Skills and Abilities', '')}"""

    # Build comprehensive prompt with two-part dialogue structure
    prompt = f"""{two_part_dialogue_prompt}

Available MNPS Roles: {', '.join(VALID_ROLES)}

{KSACS_TEXT}

Job Description to Classify:
{job_text}

**CRITICAL INSTRUCTIONS**:
1. First, write your PART 1 dialogue analysis (Examiner and Assistant exchange)
2. Then, provide PART 2 structured classification output
3. Base classification ONLY on job attributes (ignore any job title)
4. In your grouping_justification, you MUST explicitly state "This position aligns with the [ROLE NAME] role because..."

Return your response as a JSON object with this structure:
{{
  "dialogue_analysis": "Your complete Part 1 dialogue with Examiner and Assistant exchanges",
  "new_job_title": "Combine major_role_group + minor_sub_group + domain (e.g., 'Transportation Coordinator II')",
  "major_role_group": "One approved MNPS role from the list",
  "minor_sub_group": "I, II, III, Lead, or blank (blank for Teacher/Librarian/Counselor/Principal/Assistant Principal/Therapist/Director/Coach)",
  "grouping_justification": "Must start with 'This position aligns with the [ROLE NAME] role because...' then explain based on your dialogue analysis"
}}"""

    try:
        # Use GPT-4o for classification with retry logic
        response_data = call_llm_json_with_retry(prompt, MODEL_ID)

        # Extract and validate data
        major_role = response_data.get('major_role_group', 'Other')
        minor_role = response_data.get('minor_sub_group', '')
        dialogue = response_data.get('dialogue_analysis', '')

        # Apply enhanced post-processing
        major_role = discourage_specialist(job_text, major_role)
        major_role = distinguish_supervisor_manager(job_text, major_role)
        major_role = refine_coordinator_coach_manager(job_text, major_role)

        # Handle minor sub-group - normalize or set to blank for certain roles
        roles_requiring_blank = ['Teacher', 'Librarian', 'Counselor', 'Principal',
                                'Assistant Principal', 'Therapist', 'Psychologist',
                                'Social Worker', 'Director', 'Coach',
                                'Pathologist (Speech-Language Focused)']

        if major_role in roles_requiring_blank and minor_role != 'Lead':
            minor_role = ''
        elif minor_role:
            minor_role = normalize_minor(minor_role)
            minor_role = fix_executive_minor_sub_grouping(major_role, minor_role)

        # Generate improved job title if not properly formatted
        new_job_title = response_data.get('new_job_title', '')
        if not new_job_title:
            if minor_role:
                new_job_title = f"{major_role} {minor_role}"
            else:
                new_job_title = major_role

        # Combine dialogue and justification for complete context
        justification = response_data.get('grouping_justification', 'No justification provided')
        if dialogue:
            full_justification = f"DIALOGUE ANALYSIS:\n{dialogue}\n\nCLASSIFICATION JUSTIFICATION:\n{justification}"
        else:
            full_justification = justification

        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Description Name', ''),
            'new_job_title': new_job_title,
            'major_role_group': major_role,
            'minor_sub_group': minor_role if minor_role else '',
            'grouping_justification': full_justification,
            'model_used': MODEL_ID
        }
    except Exception as e:
        print(f"Error processing row {row_idx}: {e}")
        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Description Name', ''),
            'new_job_title': 'Error',
            'major_role_group': 'Other',
            'minor_sub_group': '',
            'grouping_justification': f'Error: {str(e)}',
            'model_used': MODEL_ID
        }

# Process all job descriptions with two-part dialogue
results = []
print("🚀 Starting batch processing with two-part dialogue approach...")

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing jobs"):
    result = process_job_description_dialogue(idx, row)
    results.append(result)

    # Add delay between requests to prevent rate limiting
    time.sleep(0.5)  # Increased delay due to longer dialogue responses

# Save results
results_df = pd.DataFrame(results)
output_path = OUTPUTS_DIR / "Job_Classifications_Batch_gpt4o_v800.csv"
results_df.to_csv(output_path, index=False)

print(f"✅ Processed {len(results)} job descriptions")
print(f"✅ Saved results to: {output_path}")
print(f"✅ Using two-part dialogue v8.0 with enhanced role distinctions")

In [ ]:
# ==== 8) Generate Summary Statistics and Examples (v8.0) ====

# Load the results
preds = results_df.copy()

# Generate summary statistics
major_counts = preds['major_role_group'].value_counts()
minor_counts = preds['minor_sub_group'].value_counts()

# Count roles that should have blank minor sub-groups
roles_requiring_blank = ['Teacher', 'Librarian', 'Counselor', 'Principal',
                        'Assistant Principal', 'Therapist', 'Psychologist',
                        'Social Worker', 'Director', 'Coach',
                        'Pathologist (Speech-Language Focused)']

blank_violations = preds[
    (preds['major_role_group'].isin(roles_requiring_blank)) &
    (preds['minor_sub_group'].notna()) &
    (preds['minor_sub_group'] != '') &
    (preds['minor_sub_group'] != 'Lead')
]

# Create summary
summary_stats = pd.DataFrame({
    'metric': [
        'total_rows',
        'unique_major_roles',
        'unique_minor_roles',
        'specialist_count',
        'blank_minor_sub_groups',
        'blank_rule_violations',
        'manager_count',
        'coordinator_count',
        'coach_count'
    ],
    'value': [
        len(preds),
        len(major_counts),
        len(minor_counts),
        int((preds['major_role_group'] == 'Specialist').sum()),
        int((preds['minor_sub_group'] == '').sum() + (preds['minor_sub_group'].isna()).sum()),
        len(blank_violations),
        int((preds['major_role_group'] == 'Manager').sum()),
        int((preds['major_role_group'] == 'Coordinator').sum()),
        int((preds['major_role_group'] == 'Coach').sum())
    ]
})

summary_path = OUTPUTS_DIR / "summary_stats_gpt4o_v800.csv"
summary_stats.to_csv(summary_path, index=False)

# Show examples of classifications
examples = preds[['source_row_index', 'job_title_original', 'new_job_title',
                  'major_role_group', 'minor_sub_group']].head(15)

examples_path = OUTPUTS_DIR / "examples_gpt4o_v800.csv"
examples.to_csv(examples_path, index=False)

# Save blank rule violations if any
if len(blank_violations) > 0:
    violations_path = OUTPUTS_DIR / "blank_rule_violations_v800.csv"
    blank_violations[['source_row_index', 'job_title_original', 'major_role_group',
                     'minor_sub_group', 'new_job_title']].to_csv(violations_path, index=False)
    print(f"⚠️ Found {len(blank_violations)} blank rule violations - saved to {violations_path}")

print("\n📊 Summary Statistics:")
print(summary_stats.to_string(index=False))

print("\n📈 Major Role Distribution:")
print(major_counts.to_string())

print("\n📈 Minor Role Distribution:")
print(minor_counts.to_string())

print("\n📋 Example Classifications:")
print(examples.to_string(index=False))

print(f"\n✅ Saved summary to: {summary_path}")
print(f"✅ Saved examples to: {examples_path}")
print(f"\n✅ Two-part dialogue v8.0 processing complete!")

In [ ]:
# ==== 9) Enhanced Quality Check and Validation ====

# Check for alignment issues between justification and selected roles
alignment_issues = []
for idx, row in preds.iterrows():
    justification = str(row['grouping_justification']).lower()
    major_role = str(row['major_role_group']).lower()

    # Check if justification mentions the selected role
    if major_role not in justification and major_role != 'other':
        alignment_issues.append({
            'row_index': row['source_row_index'],
            'major_role_group': row['major_role_group'],
            'justification_excerpt': row['grouping_justification'][:100] + '...'
        })

# Check for job title format consistency
title_format_issues = []
for idx, row in preds.iterrows():
    new_title = str(row['new_job_title'])
    major_role = str(row['major_role_group'])
    minor_role = str(row['minor_sub_group'])

    # Check if job title incorporates both major and minor roles
    if major_role.lower() not in new_title.lower() or minor_role.lower() not in new_title.lower():
        title_format_issues.append({
            'row_index': row['source_row_index'],
            'new_job_title': new_title,
            'major_role_group': major_role,
            'minor_sub_group': minor_role
        })

# Check for executive roles with "Lead" minor sub-grouping (should be rare)
executive_lead_issues = []
for idx, row in preds.iterrows():
    major_role = str(row['major_role_group'])
    minor_role = str(row['minor_sub_group'])

    if major_role in EXECUTIVE_ROLES and minor_role == 'Lead':
        executive_lead_issues.append({
            'row_index': row['source_row_index'],
            'major_role_group': major_role,
            'minor_sub_group': minor_role,
            'new_job_title': row['new_job_title']
        })

# Save quality check results
if alignment_issues:
    alignment_df = pd.DataFrame(alignment_issues)
    alignment_path = OUTPUTS_DIR / "alignment_issues_gpt4o_v753.csv"
    alignment_df.to_csv(alignment_path, index=False)
    print(f"⚠️  Found {len(alignment_issues)} alignment issues - saved to {alignment_path}")
else:
    print("✅ No alignment issues found")

if title_format_issues:
    title_format_df = pd.DataFrame(title_format_issues)
    title_format_path = OUTPUTS_DIR / "title_format_issues_gpt4o_v753.csv"
    title_format_df.to_csv(title_format_path, index=False)
    print(f"⚠️  Found {len(title_format_issues)} title format issues - saved to {title_format_path}")
else:
    print("✅ No title format issues found")

if executive_lead_issues:
    executive_lead_df = pd.DataFrame(executive_lead_issues)
    executive_lead_path = OUTPUTS_DIR / "executive_lead_issues_gpt4o_v753.csv"
    executive_lead_df.to_csv(executive_lead_path, index=False)
    print(f"⚠️  Found {len(executive_lead_issues)} executive roles with 'Lead' minor sub-grouping - saved to {executive_lead_path}")
else:
    print("✅ No executive roles with inappropriate 'Lead' minor sub-grouping found")

print("\n✅ Enhanced quality check completed")
